In [1]:
# =========================================================
# PSEUDO-BULK (patient-level) version of your pipeline
# - Input: per-celltype h5ad (like RR_HC_Erythroid.h5ad etc.)
# - Output: SAME style as your current script:
#     {cluster_name}__full_all_ok_genes.csv
#     {cluster_name}__SIG_up_in_{nres_key}.csv
#     ALL_clusters__SIG_up_in_{last_nres_key}__combined.csv
#     ALL_clusters__SIG_up_in_{last_nres_key}__gene_list_per_cluster.csv
#
# Key change:
#   - Instead of cell-level positive-only MWU, we:
#       1) build pseudo-bulk per patient (sum counts or mean logexpr)
#       2) run patient-level MWU per gene (n = #patients, not #cells)
#   - This usually reduces "sig genes explosion".
# =========================================================

import os
import scanpy as sc
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import mannwhitneyu


In [22]:

# =========================================================
# 0) paths
# =========================================================
DATA_DIR = "./adatas/RRErythroid"

group_files = {
    "HC_Erythroid":        os.path.join(DATA_DIR, "RR_HC_Erythroid.h5ad"),
    "post_nres_Erythroid": os.path.join(DATA_DIR, "RR_post_nres_Erythroid.h5ad"),
    "pre_res_Erythroid":   os.path.join(DATA_DIR, "RR_pre_res_Erythroid.h5ad"),
    "pre_nres_Erythroid":  os.path.join(DATA_DIR, "RR_pre_nres_Erythroid.h5ad"),
}

clusters_to_test = [
    ("HCvspost_nres", "HC_Erythroid", "post_nres_Erythroid"),
    ("HCvspre_nres",  "HC_Erythroid", "pre_nres_Erythroid"),
    ("HCvspre_res",   "HC_Erythroid", "pre_res_Erythroid"),
]


In [ ]:

# =====================
# 1) params
# =====================
PATIENT_COL = "patient"

# 你的 h5ad 没有 counts/raw，所以用 mean_logexpr
PB_MODE = "mean_logexpr"        # 固定用这个
USE_LAYER = None               # 用 adata.X；如果你想用 layers["hvg_log1p"]，改成 "hvg_log1p"
USE_RAW = False
PB_NORMALIZE = False           # mean_logexpr 下不做 CPM/log1p

MIN_CELLS_PER_PATIENT = 30     # 每个病人至少多少细胞；如果过滤后病人太少，先降到 10

# patient-level positive-only（在 patient 均值上）
THRESHOLD = 0.0
MIN_POS_PATIENTS = 2           # 建议先设 2/3；如果病人很少先用 2
FC_METHOD = "mean"
FC_PSEUDOCOUNT = 1e-9

USE_FDR = False
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05

OUTDIR = "DE_pseudobulk_patient_level_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)


In [18]:
def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def pseudobulk_patient_matrix_mean_logexpr(
    adata: sc.AnnData,
    patient_col: str,
    min_cells_per_patient: int,
    use_layer: str | None = None,
    use_raw: bool = False,
):
    """
    Pseudo-bulk by patient: mean of expression (log-scale recommended) per patient.
    Returns:
      pb_df: patients x genes
      meta_df: patient-level metadata (n_cells)
    """
    if patient_col not in adata.obs.columns:
        raise KeyError(f"'{patient_col}' not in adata.obs")

    X, genes = get_matrix_and_genes(adata, use_layer=use_layer, use_raw=use_raw)
    if X is None:
        raise ValueError("Expression matrix is None; check USE_LAYER/USE_RAW.")

    patients = adata.obs[patient_col].astype(str).values
    vc = pd.Series(patients).value_counts()
    keep_patients = vc[vc >= min_cells_per_patient].index.astype(str).tolist()

    keep_mask = np.isin(patients, keep_patients)
    if keep_mask.sum() == 0:
        raise ValueError(f"No patients with >= {min_cells_per_patient} cells.")

    X = X[keep_mask, :]
    patients = patients[keep_mask]

    uniq_patients = pd.Index(pd.unique(patients)).astype(str)
    pid_to_row = {p: i for i, p in enumerate(uniq_patients)}
    n_pat = len(uniq_patients)
    n_gene = len(genes)

    pb = np.zeros((n_pat, n_gene), dtype=float)
    counts = np.zeros(n_pat, dtype=int)

    if sparse.issparse(X):
        X = X.tocsr()

    for i, p in enumerate(patients):
        r = pid_to_row[p]
        if sparse.issparse(X):
            pb[r, :] += X[i, :].toarray().ravel()
        else:
            pb[r, :] += np.asarray(X[i, :]).ravel()
        counts[r] += 1

    pb = pb / np.maximum(counts[:, None], 1)
    pb_df = pd.DataFrame(pb, index=uniq_patients, columns=genes)

    meta_df = pd.DataFrame({
        "patient": uniq_patients,
        "n_cells": [int((patients == p).sum()) for p in uniq_patients],
    }).set_index("patient")

    return pb_df, meta_df



def patient_level_mwu_allgenes(
    pb_res: pd.DataFrame, pb_nres: pd.DataFrame,
    res_name: str, nres_name: str,
    threshold: float = 0.0,
    min_pos_patients: int = 2,
    fc_method: str = "mean",
    fc_pseudocount: float = 1e-9,
):
    common = np.intersect1d(pb_res.columns.values, pb_nres.columns.values, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    A = pb_res[common]
    B = pb_nres[common]

    rows = []
    for gene in common:
        a_all = A[gene].to_numpy(dtype=float)
        b_all = B[gene].to_numpy(dtype=float)

        a_pos = a_all[a_all > threshold]
        b_pos = b_all[b_all > threshold]

        if (a_pos.size < min_pos_patients) or (b_pos.size < min_pos_patients):
            continue

        u, p = mannwhitneyu(a_pos, b_pos, alternative="two-sided")

        stat_a = summary_stat(a_pos, method=fc_method)
        stat_b = summary_stat(b_pos, method=fc_method)

        denom = stat_a + fc_pseudocount
        numer = stat_b + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan
        delta = stat_b - stat_a

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos_patients": int(a_pos.size),
            f"{nres_name}_n_pos_patients": int(b_pos.size),
            f"{fc_method}_g1_pos": float(stat_a),
            f"{fc_method}_g2_pos": float(stat_b),
            "FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            "log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            "mean_pos_diff(g1-g2)": float(np.mean(a_pos) - np.mean(b_pos)),
            "delta_logexpr_g2_minus_g1": float(delta),
        })

    return pd.DataFrame(rows)


In [23]:

# =====================
# 3) read
# =====================
adatas = {}
for k, fp in group_files.items():
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =====================
# 4) main loop (same output style)
# =====================
all_sig_tables = []
log2fc_by_cluster = {}
sig_gene_sets = {}
_last_nres_key = None

for cluster_name, res_key, nres_key in clusters_to_test:
    _last_nres_key = nres_key

    pb_res, meta_res = pseudobulk_patient_matrix_mean_logexpr(
        adatas[res_key],
        patient_col=PATIENT_COL,
        min_cells_per_patient=MIN_CELLS_PER_PATIENT,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
    )
    pb_nres, meta_nres = pseudobulk_patient_matrix_mean_logexpr(
        adatas[nres_key],
        patient_col=PATIENT_COL,
        min_cells_per_patient=MIN_CELLS_PER_PATIENT,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
    )

    print(f"[{cluster_name}] patients: {res_key}={pb_res.shape[0]}  {nres_key}={pb_nres.shape[0]}")

    # optional save pseudo-bulk matrices
    pb_res.to_csv(os.path.join(OUTDIR, f"{cluster_name}__{res_key}__pseudobulk_patient_by_gene.csv"))
    pb_nres.to_csv(os.path.join(OUTDIR, f"{cluster_name}__{nres_key}__pseudobulk_patient_by_gene.csv"))
    meta_res.to_csv(os.path.join(OUTDIR, f"{cluster_name}__{res_key}__pseudobulk_patient_meta.csv"))
    meta_nres.to_csv(os.path.join(OUTDIR, f"{cluster_name}__{nres_key}__pseudobulk_patient_meta.csv"))

    df = patient_level_mwu_allgenes(
        pb_res, pb_nres,
        res_name=res_key, nres_name=nres_key,
        threshold=THRESHOLD,
        min_pos_patients=MIN_POS_PATIENTS,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {cluster_name}: no genes passed MIN_POS_PATIENTS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    if USE_FDR:
        sig_mask = df["FDR_BH"].notna() & (df["FDR_BH"] < FDR_CUTOFF)
    else:
        sig_mask = df["p_value"].notna() & (df["p_value"] < P_CUTOFF)

    # up_mask = df["log2FC_g2_over_g1"].notna() & (df["log2FC_g2_over_g1"] > 0)
    # sig_up = df[sig_mask & up_mask].copy()
    # sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # sig_mask = df["p_value"].notna() & (df["p_value"] < 0.05)   # 或 0.1
    # up_mask  = df["delta_logexpr_g2_minus_g1"].notna() & (df["delta_logexpr_g2_minus_g1"] > 0.25)  # 0.25~0.5
    # sig_up = df[sig_mask & up_mask].copy()

    #“Given the extremely limited number of patients (n=3 vs n=4), p-values from MWU tests are discrete with a minimum attainable value of 0.057. 
    # Therefore, we prioritized effect size (difference in mean log-expression) and used the minimum attainable p-value as a ranking criterion to 
    # identify robust candidate genes.
    min_p = df["p_value"].min()
    sig_mask = df["p_value"] <= (min_p + 1e-12)
    up_mask  = df["delta_logexpr_g2_minus_g1"] > df["delta_logexpr_g2_minus_g1"].quantile(0.95)  # df["delta_logexpr"].quantile([0.9, 0.95, 0.99])
    sig_up = df[sig_mask & up_mask].copy()


    out_full = os.path.join(OUTDIR, f"{cluster_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{cluster_name}__SIG_up_in_{nres_key}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{cluster_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    sig_up["cluster"] = cluster_name
    all_sig_tables.append(sig_up)
    log2fc_by_cluster[cluster_name] = df.set_index("gene")["log2FC_g2_over_g1"]
    sig_gene_sets[cluster_name] = set(sig_up["gene"].tolist())

# =====================
# 5) combined outputs
# =====================
if len(all_sig_tables) > 0 and _last_nres_key is not None:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(
        os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{_last_nres_key}__combined.csv"),
        index=False
    )

    list_rows = []
    for cluster_name, genes in sig_gene_sets.items():
        list_rows.append({
            "cluster": cluster_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_target": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(
        os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{_last_nres_key}__gene_list_per_cluster.csv"),
        index=False
    )

print("Done.")

[HCvspost_nres] patients: HC_Erythroid=3  post_nres_Erythroid=4
[HCvspost_nres] ok_genes=11591  sig_up=478
  saved: DE_pseudobulk_patient_level_by_cluster/HCvspost_nres__SIG_up_in_post_nres_Erythroid.csv
[HCvspre_nres] patients: HC_Erythroid=3  pre_nres_Erythroid=2
[HCvspre_nres] ok_genes=11101  sig_up=523
  saved: DE_pseudobulk_patient_level_by_cluster/HCvspre_nres__SIG_up_in_pre_nres_Erythroid.csv
[HCvspre_res] patients: HC_Erythroid=3  pre_res_Erythroid=2
[HCvspre_res] ok_genes=6631  sig_up=328
  saved: DE_pseudobulk_patient_level_by_cluster/HCvspre_res__SIG_up_in_pre_res_Erythroid.csv
Done.


In [20]:
# 1) 看 p 值最小能到哪
print(df["p_value"].min(), df["p_value"].quantile([0.01, 0.05, 0.1]).to_dict())

# 2) 看效应量分布（如果你改成 delta_logexpr）
print(df["delta_logexpr_g2_minus_g1"].quantile([0.9,0.95,0.99]).to_dict())


0.05714285714285714 {0.01: 0.05714285714285714, 0.05: 0.05714285714285714, 0.1: 0.05714285714285714}
{0.9: 0.150978307286824, 0.95: 0.2682273279770627, 0.99: 0.6389926706634893}


In [26]:
ls

ALL_clusters__SIG_up_in_pre_res_Erythroid__combined.csv
ALL_clusters__SIG_up_in_pre_res_Erythroid__gene_list_per_cluster.csv
HCvspost_nres__full_all_ok_genes.csv
HCvspost_nres__HC_Erythroid__pseudobulk_patient_by_gene.csv
HCvspost_nres__HC_Erythroid__pseudobulk_patient_meta.csv
HCvspost_nres__post_nres_Erythroid__pseudobulk_patient_by_gene.csv
HCvspost_nres__post_nres_Erythroid__pseudobulk_patient_meta.csv
HCvspost_nres__SIG_up_in_post_nres_Erythroid.csv
HCvspre_nres__full_all_ok_genes.csv
HCvspre_nres__HC_Erythroid__pseudobulk_patient_by_gene.csv
HCvspre_nres__HC_Erythroid__pseudobulk_patient_meta.csv
HCvspre_nres__pre_nres_Erythroid__pseudobulk_patient_by_gene.csv
HCvspre_nres__pre_nres_Erythroid__pseudobulk_patient_meta.csv
HCvspre_nres__SIG_up_in_pre_nres_Erythroid.csv
HCvspre_res__full_all_ok_genes.csv
HCvspre_res__HC_Erythroid__pseudobulk_patient_by_gene.csv
HCvspre_res__HC_Erythroid__pseudobulk_patient_meta.csv
HCvspre_res__pre_res_Erythroid__pseudobulk_patient_by_gene.csv
HCvsp

In [27]:


# ===== 1. 读入数据 =====
# df = pd.read_csv("../../Gao/RR/Upregulated in Post-nonresponder compared to pre-nonresponder0.csv")
df = pd.read_csv("./ALL_clusters__SIG_up_in_pre_res_Erythroid__combined.csv")

# 如果列名和你实际的不完全一致，可以在这里统一
df = df.rename(columns={
    "Gene": "gene",          # 如果原来是 Gene
    "cluster": "pair",      # 如果原来不是 pair
})

# ===== 2. 只保留你关心的 pair（可选，但推荐）=====
pairs_keep = ["HCvspost_nres", "HCvspre_nres", "HCvspre_res"]
df = df[df["pair"].isin(pairs_keep)]

# ===== 3. 设置多级索引：gene × pair =====
df_idx = df.set_index(["gene", "pair"])

# ===== 4. 选择要展开的指标列 =====
# value_cols = [
#     "p_value",
#     "mean_pre_pos",
#     "mean_post_pos",
#     "FC_post_over_pre"
# ]
value_cols = [
    "p_value",
    "mean_g1_pos",
    "mean_g2_pos",
    "FC_g2_over_g1"
]
# ===== 5. unstack pair → 自动补 NaN =====
df_wide = (
    df_idx[value_cols]
    .unstack("pair")        # columns 变成 (metric, pair)
)

# ===== 6. 调整列顺序：pair 在前，metric 在后 =====
df_wide = df_wide.swaplevel(0, 1, axis=1)
df_wide = df_wide.sort_index(axis=1, level=0)

# ===== 7. 压平成单层列名（推荐）=====
df_wide.columns = [
    f"{pair}__{metric}"
    for pair, metric in df_wide.columns
]
df_wide.to_csv('Upregulated in Pre-nonresponder compared to pre-responder.csv')